# 03 - DML, History e Time Travel em Delta Lake

Este notebook demonstra operacoes transacionais em tabelas Delta Lake armazenadas no bucket **bronze** do MinIO.

A demonstracao principal usa a tabela `produtos` e o produto de teste `id = 999` para mostrar:

- `INSERT`
- `UPDATE`
- `DELETE`
- `DESCRIBE HISTORY`
- `TIME TRAVEL` com `versionAsOf`

## 1. Configuracao e SparkSession

In [1]:
import os

from delta.tables import DeltaTable
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv(override=True)

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BRONZE_BUCKET = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

tables = ['clientes', 'produtos', 'pedidos', 'itens_pedido']
produto_teste_id = 999

spark = (
    SparkSession.builder
    .appName('DML Delta Lake Bronze')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession criada com suporte a Delta Lake e MinIO.')
print(f'Bucket bronze: {BRONZE_BUCKET}')

26/05/05 20:10:29 WARN Utils: Your hostname, And resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 20:10:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/anderson/git-clone/spark-delta-minio/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/anderson/.ivy2/cache
The jars for the packages stored in: /home/anderson/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-66c24408-90bd-4256-a1d4-23f7e0581e8e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 354ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.

SparkSession criada com suporte a Delta Lake e MinIO.
Bucket bronze: bronze


## 2. Registrar tabelas Delta do bronze no Spark SQL

In [2]:
for table in tables:
    delta_path = f's3a://{BRONZE_BUCKET}/{table}'
    spark.sql(f'DROP TABLE IF EXISTS {table}')
    spark.sql(f"""
        CREATE TABLE {table}
        USING delta
        LOCATION '{delta_path}'
    """)

print('Tabelas Delta registradas no Spark SQL:')
spark.sql('SHOW TABLES').show(truncate=False)

26/05/05 20:10:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Tabelas Delta registradas no Spark SQL:
+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|default  |clientes    |false      |
|default  |itens_pedido|false      |
|default  |pedidos     |false      |
|default  |produtos    |false      |
+---------+------------+-----------+



## 3. Leitura inicial das tabelas Delta

In [3]:
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)

for table in tables:
    count = spark.sql(f'SELECT COUNT(*) AS total FROM {table}').collect()[0]['total']
    print(f'{table:<15} {count:>10}')

print('\nAmostra da tabela produtos:')
spark.sql('SELECT * FROM produtos ORDER BY id LIMIT 10').show(truncate=False)

Tabela           Registros
---------------------------


26/05/05 20:10:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

clientes               100


produtos                50


pedidos                200


itens_pedido           400

Amostra da tabela produtos:


[Stage 35:>                                                         (0 + 1) / 1]

+---+------------------------+----------------------+------+-------+-----+
|id |nome_produto            |categoria             |preco |estoque|ativo|
+---+------------------------+----------------------+------+-------+-----+
|1  |Insulated Cooler        |Outdoor               |39.99 |1      |false|
|2  |Organic Baby Carrots    |Food - Fresh Produce  |2.99  |2      |true |
|3  |Hibiscus Tea Bags       |Food - Beverages      |3.79  |3      |false|
|4  |Spinach Artichoke Dip   |Food - Snacks         |4.99  |4      |true |
|5  |Knitted Infinity Scarf  |Clothing - Accessories|29.99 |5      |true |
|6  |Spiced Pumpkin Soup     |Food - Canned Soups   |3.99  |6      |false|
|7  |Sliced Olives           |Food - Condiments     |1.99  |7      |true |
|8  |Pineapple Salsa         |Food - Condiments     |4.29  |8      |true |
|9  |Compressed Towel Tablets|Travel                |12.99 |9      |true |
|10 |Electronic Drum Kit     |Music                 |359.99|10     |false|
+---+--------------------

## 4. Preparar produto de teste

In [4]:
spark.sql(f'DELETE FROM produtos WHERE id = {produto_teste_id}')

produto_path = f's3a://{BRONZE_BUCKET}/produtos'
dt_produtos = DeltaTable.forPath(spark, produto_path)
history_before = dt_produtos.history().select('version').orderBy('version').collect()
versao_base = history_before[-1]['version']

print(f'Produto {produto_teste_id} removido caso existisse de execucoes anteriores.')
print(f'Versao base antes do INSERT: {versao_base}')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

Produto 999 removido caso existisse de execucoes anteriores.
Versao base antes do INSERT: 0
+---+------------+---------+-----+-------+-----+
|id |nome_produto|categoria|preco|estoque|ativo|
+---+------------+---------+-----+-------+-----+
+---+------------+---------+-----+-------+-----+



## 5. INSERT - inserir produto id 999

In [5]:
spark.sql(f"""
    INSERT INTO produtos VALUES
    ({produto_teste_id}, 'Produto Delta Teste', 'Demo Delta Lake', 99.90, 10, true)
""")

print('Produto id 999 inserido:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_insert = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos INSERT: {versao_insert}')

Produto id 999 inserido:


+---+-------------------+---------------+-----+-------+-----+
|id |nome_produto       |categoria      |preco|estoque|ativo|
+---+-------------------+---------------+-----+-------+-----+
|999|Produto Delta Teste|Demo Delta Lake|99.9 |10     |true |
+---+-------------------+---------------+-----+-------+-----+

Versao apos INSERT: 1


## 6. UPDATE - atualizar produto id 999

In [6]:
spark.sql(f"""
    UPDATE produtos
    SET nome_produto = 'Produto Delta Teste Atualizado',
        preco = 149.90,
        estoque = 25,
        ativo = false
    WHERE id = {produto_teste_id}
""")

print('Produto id 999 atualizado:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_update = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos UPDATE: {versao_update}')

Produto id 999 atualizado:


+---+------------------------------+---------------+-----+-------+-----+
|id |nome_produto                  |categoria      |preco|estoque|ativo|
+---+------------------------------+---------------+-----+-------+-----+
|999|Produto Delta Teste Atualizado|Demo Delta Lake|149.9|25     |false|
+---+------------------------------+---------------+-----+-------+-----+

Versao apos UPDATE: 2


## 7. DELETE - deletar produto id 999

In [7]:
spark.sql(f'DELETE FROM produtos WHERE id = {produto_teste_id}')

print('Produto id 999 apos DELETE:')
spark.sql(f'SELECT * FROM produtos WHERE id = {produto_teste_id}').show(truncate=False)

versao_delete = dt_produtos.history(1).select('version').collect()[0]['version']
print(f'Versao apos DELETE: {versao_delete}')

Produto id 999 apos DELETE:
+---+------------+---------+-----+-------+-----+
|id |nome_produto|categoria|preco|estoque|ativo|
+---+------------+---------+-----+-------+-----+
+---+------------+---------+-----+-------+-----+

Versao apos DELETE: 3


## 8. HISTORY - historico transacional da tabela produtos

In [8]:
print('Historico Delta da tabela produtos:')
spark.sql(f'DESCRIBE HISTORY delta.`{produto_path}`') \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

Historico Delta da tabela produtos:
+-------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 9. TIME TRAVEL - consultar versoes anteriores

In [9]:
print(f'Versao base ({versao_base}) - antes do produto 999:')
spark.read.format('delta') \
    .option('versionAsOf', versao_base) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao INSERT ({versao_insert}) - produto 999 inserido:')
spark.read.format('delta') \
    .option('versionAsOf', versao_insert) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao UPDATE ({versao_update}) - produto 999 atualizado:')
spark.read.format('delta') \
    .option('versionAsOf', versao_update) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

print(f'Versao DELETE ({versao_delete}) - produto 999 deletado:')
spark.read.format('delta') \
    .option('versionAsOf', versao_delete) \
    .load(produto_path) \
    .where(f'id = {produto_teste_id}') \
    .show(truncate=False)

Versao base (0) - antes do produto 999:
+---+------------+---------+-----+-------+-----+
|id |nome_produto|categoria|preco|estoque|ativo|
+---+------------+---------+-----+-------+-----+
+---+------------+---------+-----+-------+-----+

Versao INSERT (1) - produto 999 inserido:


+---+-------------------+---------------+-----+-------+-----+
|id |nome_produto       |categoria      |preco|estoque|ativo|
+---+-------------------+---------------+-----+-------+-----+
|999|Produto Delta Teste|Demo Delta Lake|99.9 |10     |true |
+---+-------------------+---------------+-----+-------+-----+

Versao UPDATE (2) - produto 999 atualizado:
+---+------------------------------+---------------+-----+-------+-----+
|id |nome_produto                  |categoria      |preco|estoque|ativo|
+---+------------------------------+---------------+-----+-------+-----+
|999|Produto Delta Teste Atualizado|Demo Delta Lake|149.9|25     |false|
+---+------------------------------+---------------+-----+-------+-----+

Versao DELETE (3) - produto 999 deletado:
+---+------------+---------+-----+-------+-----+
|id |nome_produto|categoria|preco|estoque|ativo|
+---+------------+---------+-----+-------+-----+
+---+------------+---------+-----+-------+-----+



## 10. Resumo final

In [10]:
print('=' * 70)
print('RESUMO DAS OPERACOES DELTA LAKE')
print('=' * 70)
print(f'Produto de teste: id {produto_teste_id}')
print(f'1. INSERT executado na versao {versao_insert}')
print(f'2. UPDATE executado na versao {versao_update}')
print(f'3. DELETE executado na versao {versao_delete}')
print('4. HISTORY exibiu as operacoes WRITE, UPDATE e DELETE')
print('5. TIME TRAVEL mostrou o estado da tabela antes e depois de cada operacao')
print('=' * 70)

RESUMO DAS OPERACOES DELTA LAKE
Produto de teste: id 999
1. INSERT executado na versao 1
2. UPDATE executado na versao 2
3. DELETE executado na versao 3
4. HISTORY exibiu as operacoes WRITE, UPDATE e DELETE
5. TIME TRAVEL mostrou o estado da tabela antes e depois de cada operacao


## 11. Encerrar Spark

In [11]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
